# Лабораторная работа №3
## Марковский процесс принятия решений (MDP)




In [8]:
import numpy as np
from mdp import MDP, FrozenLakeEnv


## Исходный MDP из методических указаний


In [9]:
transition_probs = {
    's0': {'a0': {'s0': 0.5, 's2': 0.5}, 'a1': {'s2': 1}},
    's1': {'a0': {'s0': 0.7, 's1': 0.1, 's2': 0.2}, 'a1': {'s1': 0.95, 's2': 0.05}},
    's2': {'a0': {'s0': 0.4, 's2': 0.6}, 'a1': {'s0': 0.3, 's1': 0.3, 's2': 0.4}}
}

rewards = {
    's1': {'a0': {'s0': +5}},
    's2': {'a1': {'s0': -1}}
}

mdp = MDP(transition_probs, rewards, initial_state='s0')


## Реализация функции Q(s,a)
Формула:
Q(s,a) = Σ P(s'|s,a) * (R(s,a,s') + γV(s'))


In [10]:
def get_action_value(mdp, state_values, state, action, gamma):
    q = 0
    for next_state in mdp.get_next_states(state, action):
        prob = mdp.get_transition_prob(state, action, next_state)
        reward = mdp.get_reward(state, action, next_state)
        q += prob * (reward + gamma * state_values[next_state])
    return q


## Реализация обновления V(s)
V(s) = max_a Q(s,a)


In [11]:
def get_new_state_value(mdp, state_values, state, gamma):
    if mdp.is_terminal(state):
        return 0
    action_values = [
        get_action_value(mdp, state_values, state, action, gamma)
        for action in mdp.get_possible_actions(state)
    ]
    return max(action_values)


## Итерации по значениям (Value Iteration)


In [12]:
gamma = 0.9
num_iter = 100
min_difference = 0.001

state_values = {s: 0 for s in mdp.get_all_states()}

for i in range(num_iter):
    new_state_values = {
        s: get_new_state_value(mdp, state_values, s, gamma)
        for s in mdp.get_all_states()
    }

    diff = max(abs(new_state_values[s] - state_values[s])
               for s in mdp.get_all_states())

    state_values = new_state_values

    if diff < min_difference:
        break

print('Final state values:', state_values)


Final state values: {'s0': 3.7810348735476405, 's1': 7.294006423867229, 's2': 4.202140275227048}


## Поиск оптимального действия


In [13]:
def get_optimal_action(mdp, state_values, state, gamma=0.9):
    if mdp.is_terminal(state):
        return None
    actions = mdp.get_possible_actions(state)
    best_action = max(actions,
                      key=lambda a: get_action_value(mdp, state_values, state, a, gamma))
    return best_action

print('Optimal actions:')
for s in mdp.get_all_states():
    print(s, '->', get_optimal_action(mdp, state_values, s))


Optimal actions:
s0 -> a1
s1 -> a0
s2 -> a1


## Тестирование на FrozenLake


In [14]:
def value_iteration(mdp, state_values=None, gamma=0.9,
                    num_iter=1000, min_difference=1e-5):

    state_values = state_values or {
        s: 0 for s in mdp.get_all_states()
    }

    for i in range(num_iter):
        new_state_values = {
            s: get_new_state_value(mdp, state_values, s, gamma)
            for s in mdp.get_all_states()
        }

        diff = max(
            abs(new_state_values[s] - state_values[s])
            for s in mdp.get_all_states()
        )

        state_values = new_state_values

        if diff < min_difference:
            break

    return state_values


# Запуск Value Iteration на FrozenLake
mdp = FrozenLakeEnv(slip_chance=0)
state_values = value_iteration(mdp)

# Массовое тестирование
total_rewards = []

for game_i in range(1000):
    s = mdp.reset()
    rewards = []

    for t in range(100):
        s, r, done, _ = mdp.step(
            get_optimal_action(mdp, state_values, s, gamma)
        )
        rewards.append(r)

        if done:
            break

    total_rewards.append(np.sum(rewards))

print("Average reward:", np.mean(total_rewards))

Average reward: 1.0


## Краткий анализ проделанной работы

В данной лабораторной работе была реализована модель марковского процесса принятия решений.
Была запрограммирована формула вычисления Q(s,a) и реализован алгоритм Value Iteration.
В результате вычислена оптимальная функция ценности состояний и получена оптимальная политика.
Алгоритм был протестирован на среде FrozenLake, где продемонстрировал корректную сходимость.
Таким образом, методы динамического программирования позволяют эффективно находить
оптимальные стратегии в конечных MDP.
